# E791 $D^+\to\pi^-\pi^+\pi^+$ — $\rho(1450)$ mass/width closure test

This notebook extends the coefficient-only E791 closure test by floating the pole mass and pole width of the $\rho(1450)$ together with the complex coefficients.

Injected truth:

- $m_{\rho(1450)} = 1.465$ GeV;
- $\Gamma_{\rho(1450)} = 0.310$ GeV.

All other resonance masses and widths remain fixed. The $\rho(770)$ coefficient stays fixed to $1+0i$ to define the global magnitude/phase convention. Exactly one randomized fit is performed.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzGrid, DecayChannel, DecayModel, Minimizer, NonResonant,
    Parameter, RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()


## 1. E791 Fit-2 truth model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

RHO1450_MASS_TRUE = 1.4650
RHO1450_WIDTH_TRUE = 0.3100

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r*np.cos(phase), r*np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}


## 2. Free coefficients plus $\rho(1450)$ mass and width

Cartesian coefficients are unbounded. The dynamical parameters keep broad physical bounds to prevent negative widths and obviously unphysical pole positions.


In [ ]:
truth = {}

def free_coefficient(name):
    x0, y0 = truth_xy[name]
    truth[f"{name}.x"] = float(x0)
    truth[f"{name}.y"] = float(y0)
    return RealImag(
        Parameter.coefficient(f"{name}.x", 0.0, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", 0.0, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

rho1450_mass = Parameter.dynamics(
    "rho1450.mass", RHO1450_MASS_TRUE, owner="rho1450",
    backend_name="pole_mass", bounds=(1.20, 1.75), step=0.002,
)
rho1450_width = Parameter.dynamics(
    "rho1450.width", RHO1450_WIDTH_TRUE, owner="rho1450",
    backend_name="pole_width", bounds=(0.05, 0.70), step=0.002,
)
truth["rho1450.mass"] = RHO1450_MASS_TRUE
truth["rho1450.width"] = RHO1450_WIDTH_TRUE

components = [
    Resonance("sigma", (0,1), coefficients["sigma"], mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0,1), coefficients["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0,1), coefficients["f0_980"], mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), coefficients["rho1450"], mass=rho1450_mass, width=rho1450_width, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]

model = DecayModel(channel, components)

print("Free parameters:")
for p in model.parameters:
    if not p.fixed:
        print(f"  {p.name:18s} kind={p.kind.value:11s} bounds={p.bounds}")
print("number free =", sum(not p.fixed for p in model.parameters))


## 3. Fixed deterministic normalization grid

The integration points themselves stay fixed during minimization. When the $\rho(1450)$ mass or width changes, only the $\rho(1450)$ dynamical column and its interference-matrix row/column are reevaluated.


In [ ]:
GRID_N = 1000
norm = DalitzGrid(channel.parent_mass, channel.daughter_masses, resolution=GRID_N).sample()
print(f"normalization grid = {GRID_N} x {GRID_N} = {norm.size:,} points")


## 4. Generate the pseudo-data at the injected truth


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = model.generate_phase_space(N_POOL, seed=2000)
truth_cache_pool = model.prepare_cache(pool, norm)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity

data = weighted_resample(jax.random.key(791), pool, target_weights, N_DATA, replace=True)

print("toy events =", data.size)
print("truth normalization =", float(truth_normalization))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=110)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title("E791 Fit-2 pseudo-data")
plt.show()


## 5. Likelihood and randomized starting point

The coefficient starts are uniform in $[-2.5,2.5]$. The $\rho(1450)$ mass and width are independently randomized inside broad physical intervals. These are starting-point ranges; the Minuit limits for mass/width are those declared in the `Parameter` objects above.


In [ ]:
cache = model.prepare_cache(data, norm)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

minimizer = Minimizer(nll, model.parameters, tolerance=1e-4, verbose=2)

START_SEED = 314159
rng = np.random.default_rng(START_SEED)
start_values = {}
for p in model.parameters:
    if p.fixed:
        continue
    if p.kind.value == "coefficient":
        start_values[p.name] = float(rng.uniform(-2.5, 2.5))
    elif p.name == "rho1450.mass":
        start_values[p.name] = float(rng.uniform(1.30, 1.65))
    elif p.name == "rho1450.width":
        start_values[p.name] = float(rng.uniform(0.15, 0.50))

print(f"{'parameter':18s} {'truth':>11s} {'start':>11s} {'delta':>11s}")
for p in model.parameters:
    if not p.fixed:
        t = truth[p.name]
        s = start_values[p.name]
        print(f"{p.name:18s} {t:11.6f} {s:11.6f} {s-t:+11.6f}")

print("NLL(truth) =", float(nll(truth)))
print("NLL(start) =", float(nll(start_values)))


## 6. Check the JAX gradient before minimization


In [ ]:
gradient_check = minimizer.check_gradient(
    start_values,
    step_scale=1e-5,
    print_table=True,
)


## 7. Perform exactly one fit


In [ ]:
result = minimizer.fit(start_values=start_values, simplex=False, ncall=100000)

fit_values = {p.name: float(result.values[p.name]) for p in model.parameters if not p.fixed}

print("valid          =", bool(result.valid))
print("NLL(start)     =", float(nll(start_values)))
print("NLL(truth)     =", float(nll(truth)))
print("NLL(fit)       =", float(result.fval))
print("fit-truth NLL  =", float(result.fval - nll(truth)))
print("EDM            =", float(result.fmin.edm))
print("function calls =", int(result.nfcn))


## 8. Full closure table


In [ ]:
rows = []
print(f"{'parameter':18s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed:
        continue
    t = float(truth[p.name]); s = float(start_values[p.name])
    f = float(result.values[p.name]); e = float(result.errors[p.name])
    pull = (f-t)/e
    rows.append((p.name,t,s,f,e,pull))
    print(f"{p.name:18s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 9. Focus on $\rho(1450)$ mass and width


In [ ]:
dynamic_names = ["rho1450.mass", "rho1450.width"]
x = np.arange(2)
truth_dyn = np.array([truth[n] for n in dynamic_names])
start_dyn = np.array([start_values[n] for n in dynamic_names])
fit_dyn = np.array([result.values[n] for n in dynamic_names], dtype=float)
err_dyn = np.array([result.errors[n] for n in dynamic_names], dtype=float)

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.scatter(x, truth_dyn, marker="x", s=90, label="truth")
ax.scatter(x, start_dyn, s=45, label="random start")
ax.errorbar(x, fit_dyn, yerr=err_dyn, fmt=".", capsize=4, label="fit")
ax.set_xticks(x)
ax.set_xticklabels([r"$m_{\rho(1450)}$ [GeV]", r"$\Gamma_{\rho(1450)}$ [GeV]"])
ax.set_title(r"$\rho(1450)$ dynamical-parameter closure")
ax.legend()
plt.show()


## 10. Profile-like NLL scans around the fitted mass and width

These are simple one-dimensional scans with all other parameters fixed at the fitted values. They are not Minos/profile-likelihood intervals, but they make the local curvature and the position of the injected truth visible.


In [ ]:
def scan_parameter(name, values):
    base = dict(fit_values)
    out = []
    for value in values:
        point = dict(base)
        point[name] = float(value)
        out.append(float(nll(point)))
    out = np.asarray(out)
    return out - out.min()

mass_values = np.linspace(max(1.20, fit_values["rho1450.mass"]-0.12), min(1.75, fit_values["rho1450.mass"]+0.12), 120)
width_values = np.linspace(max(0.05, fit_values["rho1450.width"]-0.16), min(0.70, fit_values["rho1450.width"]+0.16), 120)

dmass = scan_parameter("rho1450.mass", mass_values)
dwidth = scan_parameter("rho1450.width", width_values)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(mass_values, dmass)
ax.axvline(RHO1450_MASS_TRUE, linestyle="--", label="truth")
ax.axvline(fit_values["rho1450.mass"], linestyle=":", label="fit")
ax.set_xlabel(r"$m_{\rho(1450)}$ [GeV]")
ax.set_ylabel(r"$\Delta$NLL")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(width_values, dwidth)
ax.axvline(RHO1450_WIDTH_TRUE, linestyle="--", label="truth")
ax.axvline(fit_values["rho1450.width"], linestyle=":", label="fit")
ax.set_xlabel(r"$\Gamma_{\rho(1450)}$ [GeV]")
ax.set_ylabel(r"$\Delta$NLL")
ax.legend()
plt.show()


## 11. Projection before and after the fit


In [ ]:
projection_cache = model.prepare_cache(pool, norm)

def projection(values, bins):
    intensity, _ = projection_cache.evaluate(values)
    w = np.asarray(pool.weights * intensity)
    h12, _ = np.histogram(np.asarray(pool.s12), bins=bins, weights=w)
    h13, _ = np.histogram(np.asarray(pool.s13), bins=bins, weights=w)
    return h12+h13

sdata = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(sdata.min(), sdata.max(), 110)
centres = 0.5*(bins[:-1]+bins[1:])
hd,_ = np.histogram(sdata,bins=bins)
hs = projection(start_values,bins); ht = projection(truth,bins); hf = projection(fit_values,bins)
for h in (hs,ht,hf): h *= hd.sum()/h.sum()

fig, ax = plt.subplots(figsize=(10,5.5))
ax.errorbar(centres,hd,yerr=np.sqrt(np.maximum(hd,1)),fmt=".",label="toy")
ax.step(centres,hs,where="mid",label="random start")
ax.step(centres,hf,where="mid",label="fit")
ax.step(centres,ht,where="mid",linestyle="--",label="truth")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.legend()
plt.show()


## Interpretation

This test checks the first genuinely dynamics-aware fit. Success requires not only coefficient closure, but also recovery of the injected $\rho(1450)$ pole mass and width with sensible uncertainties and a fitted NLL at least competitive with the injected point.

Because mass and width change the lineshape, the $\rho(1450)$ basis amplitude cannot remain numerically cached. The fitter reevaluates only that component and the corresponding interference integrals while retaining all unaffected components.
